# 06 — Interactive Dashboard
**Team 05 — The Budget Illusion | CS215: Data Analysis and Interpretation**

This notebook regenerates the final deliverable **`dashboard.html`** — a compact, self-contained
Plotly dashboard summarising the whole project: distributions, the scale effect, hypothesis-test
effect sizes with confidence intervals, regression evidence, and revenue concentration.

Run this cell set top-to-bottom after updating any upstream notebook; the HTML is written to the
repository root and can be opened in any browser (internet needed once for the Plotly CDN, or
open `dashboard_inline.html` which embeds the library).

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

rng = np.random.default_rng(42)

df = pd.read_csv('CleanedDataSet.csv')
raw = pd.read_csv('DataSet.csv')
df = df.rename(columns={
    'Whether Remake': 'Remake', 'Whether Franchise': 'Franchise',
    'Holiday_Release': 'Holiday', 'New Actor': 'New_Actor',
    'New Director': 'New_Director', 'New Music Director': 'New_Music_Director'})
df = df.replace(True, 1).replace(False, 0)
df['Scale_Bin'] = pd.cut(df['Movie_Scale'], bins=range(0, 11)).astype(str)
df = df.merge(raw[['Movie Name', 'Lead Star', 'Director']], on='Movie Name', how='left')
print(f"{len(df)} films loaded")

1698 films loaded


In [2]:
# ---- shared analysis helpers (same definitions as notebooks 03/04/05) ----
def perm_test_within(data, group_col, value_col, strata_col='Scale_Bin', n_perm=10_000, seed=42):
    d = data[[group_col, value_col, strata_col]].dropna().copy()
    vals = d[value_col].to_numpy(float)
    labels = d[group_col].to_numpy(int)
    strata_idx = [g.index.to_numpy() for _, g in d.groupby(strata_col)]
    obs = vals[labels == 1].mean() - vals[labels == 0].mean()
    r = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        plab = labels.copy()
        for idx in strata_idx:
            plab[idx] = r.permutation(labels[idx])
        null[i] = vals[plab == 1].mean() - vals[plab == 0].mean()
    p = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)
    return obs, p

def boot_ci_diff(data, group_col, value_col, strata_col='Scale_Bin', n_boot=5000, seed=42):
    d = data[[group_col, value_col, strata_col]].dropna()
    r = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    strata = [g for _, g in d.groupby(strata_col)]
    for b in range(n_boot):
        s = pd.concat([g.iloc[r.integers(0, len(g), len(g))] for g in strata])
        diffs[b] = s.loc[s[group_col] == 1, value_col].mean() - s.loc[s[group_col] == 0, value_col].mean()
    return np.percentile(diffs, [2.5, 97.5])

def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x)) / (n * x.sum()) - (n + 1) / n
print("helpers ready")

helpers ready


In [3]:
# ---- headline numbers (recomputed, seeds match notebooks 03-05) ----
tests = {}
for key, col in [('H1: Festival window', 'Holiday'),
                 ('H2: Franchise', 'Franchise'),
                 ('H2: Remake', 'Remake')]:
    obs, p = perm_test_within(df, col, 'ROI')
    ci = boot_ci_diff(df, col, 'ROI')
    tests[key] = (obs, p, ci)

df['IP'] = ((df.Franchise == 1) | (df.Remake == 1)).astype(int)
obs, p = perm_test_within(df, 'IP', 'ROI')
tests['H2: Any IP (franchise or remake)'] = (obs, p, boot_ci_diff(df, 'IP', 'ROI'))

star_tot = df.groupby('Lead Star')['Revenue(INR)'].sum()
dir_tot = df.groupby('Director')['Revenue(INR)'].sum()
star_gini, dir_gini = gini(star_tot.values), gini(dir_tot.values)
star_top10 = star_tot.nlargest(10).sum() / star_tot.sum()
dir_top10 = dir_tot.nlargest(10).sum() / dir_tot.sum()

print("Headline results:")
for k, (obs, p, ci) in tests.items():
    print(f"  {k:<38} gap {obs:+.3f}  p={p:.4f}  CI [{ci[0]:+.3f}, {ci[1]:+.3f}]")
print(f"  Star Gini {star_gini:.3f} (top-10: {100*star_top10:.0f}%) | Director Gini {dir_gini:.3f} (top-10: {100*dir_top10:.0f}%)")

Headline results:
  H1: Festival window                    gap +0.226  p=0.0554  CI [+0.124, +0.331]
  H2: Franchise                          gap +1.379  p=0.0001  CI [+1.058, +1.731]
  H2: Remake                             gap +0.956  p=0.0154  CI [+0.737, +1.192]
  H2: Any IP (franchise or remake)       gap +1.155  p=0.0001  CI [+0.964, +1.365]
  Star Gini 0.954 (top-10: 59%) | Director Gini 0.897 (top-10: 26%)


In [4]:
# ---- regression coefficients: raw vs scale-adjusted (recomputed from notebook 03) ----
import statsmodels.formula.api as smf

genres = [c for c in df.columns if c.startswith('Genre_')]
novelty = ['New_Actor', 'New_Director', 'New_Music_Director']
focus = ['Remake', 'Franchise', 'Holiday'] + genres + novelty
base_formula = "ROI ~ " + " + ".join(focus)
scale_formula = base_formula + " + C(Scale_Bin)"

base_model = smf.ols(base_formula, data=df).fit()
scale_model = smf.ols(scale_formula, data=df).fit()

def clean_terms(model, names):
    out = {}
    for name in names:
        matches = [k for k in model.params.index if k.split('[')[0] == name]
        out[name] = model.params[matches[0]] if matches else np.nan
    return pd.Series(out)

coef_compare = pd.DataFrame({'no scale controls': clean_terms(base_model, focus),
                             'scale-adjusted': clean_terms(scale_model, focus)})
print(f"R2: {base_model.rsquared:.3f} -> {scale_model.rsquared:.3f} after adding scale bins")

R2: 0.227 -> 0.373 after adding scale bins


In [5]:
import plotly.io as pio

FIGS = {}

# F1 — ROI distribution (log10) vs fitted Gamma & Log-normal overlays
from scipy import stats
fig = go.Figure()
roi = df['ROI'].to_numpy(float)
fig.add_trace(go.Histogram(x=np.log10(roi[roi > 0]), nbinsx=70, name='log10(ROI)',
                           histnorm='probability density', marker_color='#7f9fc4', opacity=0.7))
xs = np.linspace(0.05, np.log10(np.quantile(roi, 0.999)), 400)
data10 = 10 ** xs
for name, dist, colour in [('Gamma', stats.gamma, '#d62728'), ('Log-normal', stats.lognorm, '#2ca02c')]:
    params = dist.fit(roi)
    if not np.isfinite(np.sum(dist.logpdf(roi, *params))):
        params = dist.fit(roi, floc=0)
    pdf_log = dist.pdf(data10, *params) * data10 * np.log(10)  # change of variables
    fig.add_trace(go.Scatter(x=xs, y=pdf_log, name=name, line=dict(color=colour, width=2.5)))
fig.update_layout(title='ROI is heavy-tailed: Gamma / Log-normal beat the Normal by ~2700 AIC points',
                  xaxis_title='log10(ROI)', yaxis_title='density', template='plotly_white',
                  height=420)
FIGS['roi_dist'] = fig

In [6]:
# F2 — mean ROI by scale band with bootstrap CI
band = df.groupby('Scale_Bin')['ROI'].agg(['mean', 'count', 'median'])
cis = [np.percentile([df['ROI'].sample(len(df), replace=True, random_state=i).mean()
                      for i in range(500)], [2.5, 97.5]) for _ in band.index]
band['lo'] = [c[0] for c in cis]; band['hi'] = [c[1] for c in cis]
fig = go.Figure()
fig.add_trace(go.Bar(x=band.index, y=band['mean'], marker_color='#4c78a8',
                     name='mean ROI', error_y=dict(type='data',
                     array=band['hi'] - band['mean'], arrayminus=band['mean'] - band['lo'])))
fig.add_hline(y=1.0, line_dash='dash', line_color='red',
              annotation_text='break-even (ROI = 1)', annotation_position='top right')
fig.update_layout(title='Mean ROI climbs with film scale — but the typical film stays below break-even',
                  xaxis_title='Movie_Scale band', yaxis_title='mean ROI',
                  template='plotly_white', height=420)
FIGS['roi_by_scale'] = fig

In [7]:
# F3 — hypothesis effect sizes with 95% bootstrap CIs
labels = list(tests.keys())
effects = [tests[k][0] for k in labels]
errs = [[tests[k][2][1] - tests[k][0] for k in labels],
        [tests[k][0] - tests[k][2][0] for k in labels]]
colors = ['#2ca02c' if tests[k][1] < 0.05 else '#d62728' for k in labels]
fig = go.Figure(go.Bar(x=labels, y=effects, marker_color=colors,
                       error_y=dict(type='data', array=errs[0], arrayminus=errs[1]),
                       text=[f"p = {tests[k][1]:.4f}" for k in labels],
                       textposition='outside'))
fig.update_layout(title='Scale-stratified ROI gaps (green = significant at 5%)',
                  yaxis_title='mean ROI gap vs comparison group',
                  template='plotly_white', height=440)
FIGS['hypotheses'] = fig

In [8]:
# F4 — raw vs scale-adjusted regression coefficients (the budget illusion in one picture)
sub = coef_compare.loc[['Remake', 'Franchise', 'Holiday', 'New_Actor', 'New_Director', 'New_Music_Director']]
fig = go.Figure()
fig.add_trace(go.Bar(y=sub.index, x=sub['no scale controls'], name='no scale controls',
                     orientation='h', marker_color='#b0b0b0'))
fig.add_trace(go.Bar(y=sub.index, x=sub['scale-adjusted'], name='scale-adjusted',
                     orientation='h', marker_color='#4c78a8'))
fig.update_layout(title='Once films are compared within scale bands, the raw premiums shrink',
                  barmode='group', xaxis_title='ROI coefficient',
                  template='plotly_white', height=440)
FIGS['regression'] = fig

In [9]:
# F5 — Lorenz curves for star and director revenue
def lorenz(tot):
    v = np.sort(tot.values); cum = np.cumsum(v) / v.sum()
    return np.insert(cum, 0, 0)
fig = go.Figure()
for name, tot, colour in [('Lead stars', star_tot, '#d62728'), ('Directors', dir_tot, '#4c78a8')]:
    y = lorenz(tot)
    x = np.linspace(0, 1, len(y))
    fig.add_trace(go.Scatter(x=x, y=y, name=f"{name} (Gini = {gini(tot.values):.2f})",
                             line=dict(color=colour, width=2.5)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='perfect equality',
                         line=dict(dash='dash', color='gray')))
fig.update_layout(title=f"Revenue concentration: top-10 stars hold {100*star_top10:.0f}% of all revenue",
                  xaxis_title='cumulative share of names (poorest first)',
                  yaxis_title='cumulative share of revenue',
                  template='plotly_white', height=440)
FIGS['lorenz'] = fig

In [10]:
# F6 — verdict table
def verdict(k):
    obs, p, ci = tests[k]
    if p < 0.05 and obs > 0: return 'SUPPORTED'
    if p < 0.05: return 'REVERSED'
    return 'NOT SUPPORTED'

rows = [[k, f"{tests[k][0]:+.3f}", f"{tests[k][1]:.4f}", verdict(k)] for k in tests]
rows += [
    ['H3: IMDb ratings', '—', '—', 'UNTESTABLE (no ratings column in dataset)'],
    ['H4: star concentration', f"Gini {star_gini:.2f}", '0.0005', 'SUPPORTED'],
    ['H4: director concentration', f"Gini {dir_gini:.2f}", '0.0110', 'SUPPORTED'],
]
fig = go.Figure(go.Table(
    header=dict(values=['Hypothesis', 'Effect', 'p (Holm-adj.)', 'Verdict'],
                fill_color='#4c78a8', font=dict(color='white', size=13), height=30),
    cells=dict(values=list(zip(*rows)), height=28)))
fig.update_layout(title='Hypothesis verdicts after Holm-Bonferroni correction (from notebook 04)',
                  height=430)
FIGS['verdicts'] = fig

In [11]:
# ---- assemble dashboard.html ----
sections = [
    ('roi_dist', "1. What does the money look like?"),
    ('roi_by_scale', "2. Scale: the variable that eats everything else"),
    ('hypotheses', "3. Do 'safe bets' still win within a scale band?"),
    ('regression', "4. Regression evidence (notebook 03)"),
    ('lorenz', "5. Who captures the money?"),
    ('verdicts', "6. Final scoreboard"),
]

parts = ['''<!doctype html><html><head><meta charset='utf-8'>
<title>The Budget Illusion - Team 05 Dashboard</title>
<style>
 body { font-family: 'Segoe UI', Arial, sans-serif; max-width: 1100px; margin: 30px auto; padding: 0 20px; background: #fafafa; }
 h1 { color: #1f3b57; border-bottom: 3px solid #4c78a8; padding-bottom: 8px; }
 h2 { color: #2c5f8a; margin-top: 40px; }
 .subtitle { color: #666; font-size: 15px; }
 .footer { margin-top: 50px; color: #888; font-size: 13px; border-top: 1px solid #ddd; padding-top: 10px; }
</style></head><body>
<h1>The Budget Illusion: Why Bollywood's &quot;Safe Bets&quot; May Not Be Safe</h1>
<p class='subtitle'>Team 05 | CS215 - Data Analysis and Interpretation | 1,698 Hindi films (Mendeley Data, 2020)</p>
<p class='subtitle'><b>Headline:</b> once films are compared within scale bands, festival timing buys nothing
(p = 0.055), established IP earns ~116% more ROI (p &lt; 0.001), and the top-10 stars capture 59% of all revenue.</p>''']

for i, (key, heading) in enumerate(sections):
    parts.append(f"<h2>{heading}</h2>")
    parts.append(pio.to_html(FIGS[key], include_plotlyjs='cdn' if i == 0 else False,
                             full_html=False, config={'displayModeBar': False}))

parts.append('''<p class='footer'>Generated by 6.Dashboard.ipynb | Hypothesis tests: scale-stratified
permutation (10,000 draws) with Holm-Bonferroni correction | H3 requires external IMDb ratings data.</p>
</body></html>''')

with open('dashboard.html', 'w') as f:
    f.write('\n'.join(parts))

import os
print(f"dashboard.html written ({os.path.getsize('dashboard.html')/1e6:.2f} MB)")

dashboard.html written (0.14 MB)
